# 台灣三城市 11 年天氣 EDA

stage2_eda（AI-8 資料分析）。資料來源：`data/raw/{taipei,taichung,kaohsiung}.csv`，
Open-Meteo Historical Weather API（ERA5 再分析網格資料，非氣象站觀測，見
`docs/DATA_SOURCES.md`），區間 2015-01-01 ~ 2025-12-31，每城 4018 天。

本 notebook 只負責「串成分析敘事」，實際的載入/清理/彙整/畫圖邏輯全部在
`stage2_eda/weather_eda.py`（有獨立 pytest 測試），這裡 import 使用。

In [1]:
import sys
from pathlib import Path

# repo 根目錄推導：不管這份 notebook 是從 repo 根目錄啟動 Jupyter（cwd=repo 根)
# 還是從 stage2_eda/ 目錄啟動（cwd=stage2_eda),都要找到同一個 repo 根目錄。
# 判斷依據：repo 根目錄底下一定有 requirements.txt。
_candidate = Path.cwd()
if not (_candidate / "requirements.txt").exists() and (_candidate.parent / "requirements.txt").exists():
    _candidate = _candidate.parent
REPO_ROOT_FOR_NOTEBOOK = _candidate
sys.path.insert(0, str(REPO_ROOT_FOR_NOTEBOOK))

import pandas as pd

from common.paths import REPO_ROOT
from stage2_eda.weather_eda import *

assert REPO_ROOT == REPO_ROOT_FOR_NOTEBOOK, "notebook 推導出的 repo 根目錄與 common.paths.REPO_ROOT 對不上"

pd.set_option("display.max_columns", 20)
df = load_all_cities()
print(f"合併後總列數：{len(df)}（預期 3 城市 x 4018 天 = 12054）")
df.head()

合併後總列數：12054（預期 3 城市 x 4018 天 = 12054）


,date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,precipitation_hours,windspeed_10m_max,windgusts_10m_max,winddirection_10m_dominant,shortwave_radiation_sum,city,year,month
0,2015-01-01,13.4,10.7,11.9,0.0,0.0,0.0,18.3,47.5,57,8.91,taipei,2015,1
1,2015-01-02,14.8,10.7,12.5,0.0,0.0,0.0,19.8,51.8,82,12.46,taipei,2015,1
2,2015-01-03,18.2,12.0,14.7,0.0,0.0,0.0,17.4,44.6,101,13.12,taipei,2015,1
3,2015-01-04,21.1,12.6,16.8,0.0,0.0,0.0,6.6,19.8,144,13.81,taipei,2015,1
4,2015-01-05,22.8,14.4,18.2,0.0,0.0,0.0,7.3,20.5,149,13.58,taipei,2015,1


## 1. 資料清理與品質檢查

先看有沒有缺值、型別是不是正確的數字型別、有沒有離群值需要留意。

In [2]:
df, report = clean_data(df)
print("缺值統計：", report["missing_values"])
print()
print("型別檢查：", report["dtypes"])
print()
print(f"總列數：{report['row_count']}")
print("離群值數量（IQR 方法，k=1.5）：", report["outlier_counts"])

缺值統計： {'temperature_2m_max': 0, 'temperature_2m_min': 0, 'temperature_2m_mean': 0, 'precipitation_sum': 0, 'rain_sum': 0, 'precipitation_hours': 0, 'windspeed_10m_max': 0, 'windgusts_10m_max': 0, 'winddirection_10m_dominant': 0, 'shortwave_radiation_sum': 0}

型別檢查： {'temperature_2m_max': 'float64', 'temperature_2m_min': 'float64', 'temperature_2m_mean': 'float64', 'precipitation_sum': 'float64', 'rain_sum': 'float64', 'precipitation_hours': 'float64', 'windspeed_10m_max': 'float64', 'windgusts_10m_max': 'float64', 'winddirection_10m_dominant': 'int64', 'shortwave_radiation_sum': 'float64'}

總列數：12054
離群值數量（IQR 方法，k=1.5）： {'temperature_2m_max': 144, 'precipitation_sum': 1550, 'windspeed_10m_max': 263}


**清理結論**：三個欄位（溫度、降雨、風速）用 IQR 方法（1.5 倍 IQR）抓出的
「離群值」筆數不算少（降雨尤其多，1550 筆），但這是預期中的結果——台灣的
降雨本來就是右偏分布（大多數天不下雨或下小雨，少數天有豪大雨），用常態
分布假設的 IQR 去看降雨，本來就會把「正常的强降雨」標成離群值。這裡**不
刪除任何一列**：這些所謂的離群值多半是真實的極端天氣事件（颱風、梅雨鋒
面），刪掉會讓分析失真，反而應該保留下來另外分析（見下面「極端事件盤點」
一節）。缺值統計全部為 0，型別檢查全部欄位都已正確解析成 `float64`/
`int64`，資料清理這一步沒有需要修正的問題。

## 2. 描述性統計

看一下三城市各數值欄位的基本統計量,對資料尺度先有個底。

In [3]:
df.groupby("city")[["temperature_2m_max", "temperature_2m_min", "precipitation_sum", "windspeed_10m_max"]].describe().round(2)

temperature_2m_max                                              \
                       count   mean   std   min   25%    50%   75%   max   
city                                                                       
kaohsiung             4018.0  27.96  3.34  10.9  25.9  28.60  30.6  35.0   
taichung              4018.0  27.30  4.35   8.2  24.7  28.40  30.5  38.2   
taipei                4018.0  26.37  5.82   6.2  22.0  26.95  31.4  38.2   

          temperature_2m_min         ... precipitation_sum         \
                       count   mean  ...               75%    max   
city                                 ...                            
kaohsiung             4018.0  22.09  ...              3.30  302.6   
taichung              4018.0  19.30  ...              5.88  251.4   
taipei                4018.0  20.24  ...              4.80  278.4   

          windspeed_10m_max                                            
                      count   mean   std  min   25%   50%   75%   max  
city                                                                   
kaohsiung            4018.0  16.75  6.40  4.7  12.6  15.5  19.2  78.8  
taichung             4018.0  14.81  6.19  4.5  10.5  13.4  17.7  74.5  
taipei               4018.0  16.09  6.03  4.0  11.6  15.9  20.0  71.7  

[3 rows x 32 columns]

台北的溫度標準差看起來會比高雄大（冬夏溫差更明顯，見下面圖一），高雄
全年偏暖；降雨量三城市的中位數都遠低於平均數，符合「大多數天氣不下雨、
少數天氣下很多雨」的右偏分布特徵，這也是為什麼上一節 IQR 抓到這麼多降雨
「離群值」。

## 3. 探索性圖表

以下六張圖依序展開,每張圖後面附一段解讀。圖表存檔在 `stage2_eda/figures/`
（committed，方便不重跑 notebook 也能看結果）。

In [4]:
import matplotlib
matplotlib.use("Agg")  # headless 執行（nbclient/CI）不需要互動視窗

# 用 common.paths.REPO_ROOT 推導（見上一格），不要用 Path.cwd() 猜——
# 猜錯的話圖表會存到 repo 外面去（這裡曾經真的踩過這個坑，見 report.md）。
FIGURES_DIR = REPO_ROOT / "stage2_eda" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f"圖表輸出目錄：{FIGURES_DIR}")

圖表輸出目錄：/Users/lvshaomin/Desktop/SideProject/weather-data-science-course/stage2_eda/figures


### 圖一：三城市溫度年週期

In [5]:
fig1 = plot_temperature_annual_cycle(df)
fig1.savefig(FIGURES_DIR / "fig01_temperature_annual_cycle.png", dpi=100)
fig1

<Figure size 800x500 with 1 Axes>

**解讀**：三城市的月平均溫都在 7 月達到高峰（三城市都超過 27°C），1 月
最低。台北和台中的冬季（1~2 月）月均溫非常接近（約 16~17°C），高雄則明顯
更暖（約 19~20°C）——符合高雄緯度較低、冬季受東北季風影響較小的地理常識。
夏季三城市差距縮小，7 月三城市都在 27~29°C 區間，顯示夏季高溫是全台性的
現象，冬季溫差才是三城市最大的分野。

### 圖二：三城市高溫比較

In [6]:
fig2 = plot_high_temp_comparison(df)
fig2.savefig(FIGURES_DIR / "fig02_high_temp_comparison.png", dpi=100)
fig2

<Figure size 800x500 with 1 Axes>

**解讀**：箱型圖顯示台北的每日最高溫分布範圍（IQR）比高雄寬——台北的
冬夏溫差大，資料更分散；高雄的箱型圖明顯更窄且中位數更高，全年高溫都
維持在相對穩定且偏暖的區間。台中則介於兩者之間。三個城市的上界（最熱的
少數天）都逼近或超過 38°C，顯示夏季極端高溫是三城共同會遇到的情況。

### 圖三：降雨季節分布

In [7]:
fig3 = plot_rainfall_seasonal_distribution(df)
fig3.savefig(FIGURES_DIR / "fig03_rainfall_seasonal_distribution.png", dpi=100)
fig3

<Figure size 800x500 with 1 Axes>

**解讀**：台北的降雨在冬季（11~2 月）明顯高於台中、高雄——這是東北季風
迎風面地形雨的典型特徵。高雄與台中則在夏季（6~8 月）降雨量明顯拉高，冬季
降雨量趨近於 0，是典型的「乾濕季分明」的西南部氣候。這張圖具體呈現了
台灣「北部冬雨、中南部夏雨」的地理課本常識，用 11 年的真實資料驗證了
這個常識不是巧合。

### 圖四：極端高溫/暴雨事件盤點

In [8]:
fig4 = plot_extreme_events(df)
fig4.savefig(FIGURES_DIR / "fig04_extreme_events.png", dpi=100)
fig4

<Figure size 800x500 with 1 Axes>

In [9]:
extreme_events_summary(df)

,extreme_hot_days,heavy_rain_days,total_days
city,,,
kaohsiung,59,74,4018
taichung,176,93,4018
taipei,508,47,4018


**解讀**：台北 11 年累計 508 天最高溫達到 33°C 以上，遠高於台中（176 天）
與高雄（59 天）——這個結果乍看違反直覺（一般印象裡高雄比台北熱），但要
注意這是「≥33°C 的天數」，不是「平均溫度」（圖一已經顯示高雄平均溫度
其實更穩定溫和，很少出現台北那種夏季悶熱極端值，加上台北盆地地形不利
散熱，容易出現都市熱島加成的極端高溫日）。暴雨天數（≥50mm）則是台中
（93 天）最高、高雄（74 天）次之、台北（47 天）最少——但單日最大降雨
前五名裡有 3 筆出現在高雄（見下方 top 5），這代表高雄雖然暴雨天數不是
三城最多，但強度可以非常極端——單一颱風或午後對流帶來的降雨量特別
集中。

In [10]:
top_extreme_events(df, "precipitation_sum", n=5)

,date,city,precipitation_sum
11529,2024-07-25,kaohsiung,302.6
11599,2024-10-03,kaohsiung,288.1
219,2015-08-08,taipei,278.4
9366,2018-08-23,kaohsiung,277.5
5986,2020-05-22,taichung,251.4


前五名單日最大降雨：高雄佔了 3 筆（含 2024-07-25 的 302.6mm，是全部
11 年 x 3 城市資料裡的最大值），另外台北、台中各一筆——這與上一段「高雄
暴雨天數少但強度極端」的解讀一致。

### 圖五：年際趨勢

In [11]:
fig5 = plot_annual_trend(df)
fig5.savefig(FIGURES_DIR / "fig05_annual_trend.png", dpi=100)
fig5

<Figure size 800x500 with 1 Axes>

**解讀**：三城市的年平均溫在 2015-2025 這 11 年間沒有呈現單調遞增的
明顯趨勢，年與年之間有起伏（例如某些年份偏暖、某些年份偏涼），這與
「暖化趨勢在短短 11 年、單一地區的年均溫上不一定能看出穩定的線性趨勢」
的氣候學常識一致——年際變異（年與年之間的自然波動）在這個時間尺度下，
通常比長期趨勢訊號更明顯。這也是一個誠實的教學點：不能因為畫了一張
「年份 x 溫度」的折線圖，就急著下「有暖化」或「沒有暖化」的結論，11 年
的樣本數對於偵測氣候尺度的趨勢來說是不足的。

### 圖六：變數相關矩陣

In [12]:
fig6 = plot_correlation_matrix(df)
fig6.savefig(FIGURES_DIR / "fig06_correlation_matrix.png", dpi=100)
fig6

<Figure size 800x700 with 2 Axes>

In [13]:
diff = df[df["precipitation_sum"] != df["rain_sum"]]
diff[["date", "city", "precipitation_sum", "rain_sum"]]

,date,city,precipitation_sum,rain_sum
387,2016-01-23,taipei,27.9,27.8
388,2016-01-24,taipei,5.6,1.6
4406,2016-01-24,taichung,9.7,8.8


**解讀**：三種溫度欄位（`temperature_2m_max/min/mean`）彼此高度相關
（0.85~0.97），這是預期中的結果（同一天的最高溫、最低溫、平均溫本來就
高度連動）。`precipitation_sum` 與 `rain_sum` 相關係數在熱圖上顯示 1.00
（四捨五入到小數點後兩位），但**實際算出來是 0.9999969，不是恰好 1.0**——
上面這格程式碼揪出真正的原因：三城市 12054 筆資料裡，剛好有 3 筆
`precipitation_sum > rain_sum`，全部集中在 **2016-01-23~24**（台北兩筆、
台中一筆）。這個日期與台灣 2016 年 1 月那波破紀錄低溫寒流的時間重疊
（當時台灣多處平地出現罕見降雪或凍雨報導），推測這 3 筆的落差就是當天
總降水量（`precipitation_sum`，包含固態/凍雨）扣掉純液態雨量
（`rain_sum`）的部分——但這裡只能說「推測」，因為 CSV 本身沒有天氣現象
代碼欄位可以直接驗證是不是真的下雪/凍雨，只能從數字落差反推。這是一個
很好的教學提醒：相關矩陣的四捨五入顯示可能掩蓋掉「幾乎但不是完全相等」
的細節，真正想確認兩欄關係要重新算出未四捨五入的數字，甚至像這樣揪出
不相等的那幾筆逐一檢視。

比較有意思的另一個訊號：`shortwave_radiation_sum`（短波輻射，日照強度的
代理指標）與溫度正相關（與 `temperature_2m_max` 相關係數 0.68），與降雨
負相關（與 `precipitation_hours` 相關係數 -0.47）——晴天日照強、氣溫高；
陰雨天日照弱、氣溫相對低，符合物理直覺。風速兩欄
（`windspeed_10m_max`/`windgusts_10m_max`）彼此相關係數 0.91，同樣是
預期中的高度連動。

## 4. 整體結論

1. **地理常識被資料驗證**：北部冬雨、中南部夏雨（圖三），高雄冬季比台北
   暖（圖一），這些教科書常識在 11 年的真實資料上都成立，不是憑印象。
2. **「熱」跟「熱的次數多」是兩件事**：高雄年均溫最穩定溫暖，但極端高溫
   （≥33°C）天數遠低於台北（圖四）——台北盆地地形容易在夏季出現更頻繁
   的極端高溫日，這比單看年均溫更能說明「台北夏天很悶熱」的體感。
3. **降雨資料是右偏分布，不能用常態分布的離群值方法直接篩掉極端值**——
   本 notebook 的清理階段刻意保留所有離群值，改用 IQR 統計量描述分布
   形狀，而非把它們當成錯誤資料處理。
4. **11 年不足以判斷氣候尺度的暖化趨勢**（圖五）——這是本分析主動指出的
   方法論限制，不是迴避，只是誠實承認樣本時間長度的侷限。
5. **ERA5 再分析網格資料 ≠ 氣象站觀測**（見 `docs/DATA_SOURCES.md`）——本
   notebook 全部分析建立在網格估計值上，局部強對流天氣（例如只下在半個
   台北市的午後雷陣雨）可能無法被這份資料完整捕捉。

以上數字全部來自這份 notebook 實際執行的輸出（headless 執行,nbclient
驗證全部 cell 無錯誤),不是手填的估計值。